In [ ]:
device = "cuda:1"

### Preliminaries

In [ ]:
import itertools
import random
import collections


import transformers
import torch
import tqdm.auto
from torch import Tensor

In [ ]:
def sinusoidal_encode(
    x: Tensor,
    embedding_dim: int,
    min_value: int,
    max_value: int,
    use_l2_norm: bool = False,
    norm_const: float | None = None,
) -> Tensor:
    """
    Encodes a tensor of numbers into a sinusoidal representation, inspired by how absolute positional
    encoding works in transformers.

    The encoding is an evaluation of a sine and cosine function at different frequencies, where the
    frequency is determined by the embedding dimension and the allowed range of the input values.

    >>> sinusoidal_encode(
    ...     torch.tensor([-5, 2, 1, 0]),
    ...     embedding_dim=6,
    ...     min_value=-5,
    ...     max_value=5,
    ... )
    tensor([[ 0.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000],
            [ 0.6570,  0.7539, -0.1073, -0.9942,  0.9980,  0.0627],
            [-0.2794,  0.9602,  0.3491, -0.9371,  0.9616,  0.2746],
            [-0.9589,  0.2837,  0.7317, -0.6816,  0.8806,  0.4738]])
    """

    if embedding_dim % 2 != 0 and not use_l2_norm:
        raise ValueError("Embedding dimension must be even")

    if use_l2_norm:
        if embedding_dim % 2 == 0:
            reserved_dim = 2
        else:
            reserved_dim = 1
        embedding_dim -= reserved_dim
    else:
        reserved_dim = 0  # will not be used

    domain = max_value - min_value
    y_shape = x.shape + (embedding_dim,)
    y = torch.zeros(y_shape, device=x.device)
    even_indices = torch.arange(0, embedding_dim, 2)
    log_term = torch.log(torch.tensor(domain)) / embedding_dim
    div_term = torch.exp(even_indices * -log_term)
    x = x - min_value
    values = x.unsqueeze(-1).float() * div_term
    y[..., 0::2] = torch.sin(values)
    y[..., 1::2] = torch.cos(values)

    if use_l2_norm:
        y = torch.cat([y, torch.ones_like(y[..., :reserved_dim])], dim=-1)
        y /= y.norm(dim=-1, keepdim=True, p=2)

    if norm_const is not None:
        y *= norm_const

    return y


def binary_encode(
    x: Tensor,
    embedding_dim: int,
    min_value: int | float,
    max_value: int | float,
    use_l2_norm: bool = False,
    norm_const: float | None = None,
) -> Tensor:
    y = torch.zeros(x.shape + (embedding_dim,), device=x.device)
    reserve_dim = 0 if not use_l2_norm else 1
    x = x - min_value
    maximum = x.max()
    for i in range(embedding_dim - reserve_dim):
        coeff = 2**i
        if maximum < coeff:
            break
        y[..., -i - 1] = torch.floor(x / coeff) % 2
        x = x - coeff * y[..., -i - 1]
    if use_l2_norm:
        y = torch.cat([y, torch.ones_like(y[..., :reserve_dim])], dim=-1)
        y /= y.norm(dim=-1, keepdim=True, p=2)
    if norm_const is not None:
        y *= norm_const
    return y

### Prepare model and data

In [ ]:
model_ckpt = "meta-llama/Llama-3.2-3B"
model = transformers.AutoModel.from_pretrained(model_ckpt).eval()
tokenizer = transformers.AutoTokenizer.from_pretrained(model_ckpt)
model = model.half().to(device).eval()

In [ ]:
all_values = torch.arange(0, 1000)
mask = torch.rand(len(all_values), generator=torch.Generator().manual_seed(0))
train_mask = mask < 0.9
valid_mask = ~train_mask & (mask < 0.95)
test_mask = ~train_mask & ~valid_mask

train_values = all_values[train_mask]
valid_values = all_values[valid_mask]
test_values = all_values[test_mask]

In [ ]:
all_inputs = [(x1, x2) for x1, x2 in itertools.product(all_values.tolist(), repeat=2) if x1 * x2 < 1000]
train_values_set = set(train_values.tolist())
valid_values_set = set(valid_values.tolist())
test_values_set = set(test_values.tolist())

all_inputs_add = [(x1, x2) for x1, x2 in itertools.product(all_values.tolist(), repeat=2) if x1 + x2 < 1000]
train_values_set = set(train_values.tolist())
valid_values_set = set(valid_values.tolist())
test_values_set = set(test_values.tolist())

train_inputs = [(x1, x2) for x1, x2 in all_inputs if x1 * x2 in train_values_set]
train_inputs_add = [(x1, x2) for x1, x2 in all_inputs_add if x1 + x2 in train_values_set]
valid_inputs = [(x1, x2) for x1, x2 in all_inputs if x1 * x2 in valid_values_set]
valid_inputs_add = [(x1, x2) for x1, x2 in all_inputs_add if x1 + x2 in valid_values_set]
test_inputs = [(x1, x2) for x1, x2 in all_inputs if x1 * x2 in test_values_set]
test_inputs_add = [(x1, x2) for x1, x2 in all_inputs_add if x1 + x2 in test_values_set]

# sanity check
assert set(train_inputs) & set(valid_inputs) == set()
assert set(train_inputs) & set(test_inputs) == set()
assert set(valid_inputs) & set(test_inputs) == set()

assert set(train_inputs_add) & set(valid_inputs_add) == set()
assert set(train_inputs_add) & set(test_inputs_add) == set()
assert set(valid_inputs_add) & set(test_inputs_add) == set()

random.seed(0)
random.shuffle(train_inputs)
random.shuffle(valid_inputs)
random.shuffle(test_inputs)

random.shuffle(train_inputs_add)
random.shuffle(valid_inputs_add)
random.shuffle(test_inputs_add)

valid_size = 4096
train_size = 50_000
train_inputs = train_inputs[:train_size]
train_inputs_add = train_inputs_add[:train_size]
valid_inputs = valid_inputs[:valid_size]

In [ ]:
max([x1 * x2 for x1, x2 in all_inputs])

In [ ]:
num_templates = 6

def make_str_input(operands: tuple[int, int] | list[int], template_idx: int = 1) -> str:
    x1, x2 = operands
    options = [
        f"{x1} times {x2} is ",
        f"{x1} multiplied by {x2} is ",
        f"{x1} multiplied by {x2} equals to ",
        f"{x1} * {x2} = ",
        f"A multiplication of {x1} and {x2} equals to ",
        f"A result of multiplying {x1} and {x2} is ",
    ]
    assert num_templates == len(options)
    # return f"{x1} times {x2} is "  # 0.78
    # return f"{x1} multiplied by {x2} is "  # 90.38
    return options[template_idx]

make_str_input((3, 500)), make_str_input((3, 0))

In [ ]:
def make_str_input_add(operands: tuple[int, int] | list[int]) -> str:
    x1, x2 = operands
    return f"{x1} plus {x2} is equal to "

make_str_input_add((3, 500)), make_str_input_add((3, 0))

In [ ]:
tokenizer('3 plus 500 is equal to ').input_ids

In [ ]:
def get_hidden_states_and_preds_old(model, str_inputs: list[str], batch_size: int) -> tuple[dict[int, Tensor], list[str]]:
    model.eval()
    hidden_states = collections.defaultdict(list)
    model_preds = []
    with torch.no_grad():
        num_batches = (len(str_inputs) + batch_size - 1) // batch_size
        for batch_str in tqdm.auto.tqdm(itertools.batched(str_inputs, n=batch_size), total=num_batches, desc="Inferring model hidden states"):
            batch_inputs = tokenizer(batch_str, return_tensors="pt")
            model_outputs = model(**batch_inputs.to(model.device), output_hidden_states=True)
            hidden_reprs = model_outputs.hidden_states
            logits = model_outputs.last_hidden_state @ model.embed_tokens.weight.T
            next_token_ids = logits[:, -1, :].argmax(dim=-1)
            model_preds.extend(tokenizer.batch_decode(next_token_ids))

            for layer_idx, hidden_state in enumerate(hidden_reprs):
                hidden_states[layer_idx].extend(hidden_state[:, -1, :].detach().cpu())
    return {k: torch.stack(v) for k, v in hidden_states.items()}, model_preds

In [ ]:
import collections
import itertools
from typing import Optional
import torch
from torch import Tensor
import tqdm.auto


def get_hidden_states_and_preds(
    model,
    str_inputs: list[str],
    batch_size: int,
    component: Optional[str] = None,
) -> tuple[dict[int, Tensor], list[str]]:

    # --- ensure tokenizer can pad (as per original code) ---
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.eval()
    preds: list[str] = []

    # always record standard hidden states; fallback if component=None
    base_hidden = collections.defaultdict(list)

    # hooks & per-component buffers
    hooks = []
    q_store = k_store = v_store = comp_store = None

    # --- setup component capture if requested ---
    if component is not None:
        valid = {"q", "k", "v", "qkv", "attn_out", "mlp_out", "layer_out"}
        if component not in valid:
            raise ValueError(f"component must be one of {sorted(valid)}, got {component!r}")

        if component in ("q", "qkv"): q_store = collections.defaultdict(list)
        if component in ("k", "qkv"): k_store = collections.defaultdict(list)
        if component in ("v", "qkv"): v_store = collections.defaultdict(list)
        if component in ("attn_out", "mlp_out", "layer_out"): comp_store = collections.defaultdict(list)

        def hook_maker(store, idx):
            def hook(_, __, out):
                out = out[0] if isinstance(out, tuple) else out
                if out.dim() > 3:  # flatten heads if needed
                    b, s = out.size(0), out.size(-2)
                    out = out.view(b, s, -1)
                store[idx].append(out[:, -1, :].detach().cpu())
            return hook

        for li, layer in enumerate(model.layers):
            idx = li + 1  # match hidden_states indexing (0=embeddings)

            if component in ("q","qkv"):
                hooks.append(layer.self_attn.q_proj.register_forward_hook(hook_maker(q_store, idx)))
            if component in ("k","qkv"):
                hooks.append(layer.self_attn.k_proj.register_forward_hook(hook_maker(k_store, idx)))
            if component in ("v","qkv"):
                hooks.append(layer.self_attn.v_proj.register_forward_hook(hook_maker(v_store, idx)))
            if component == "attn_out":
                hooks.append(layer.self_attn.register_forward_hook(hook_maker(comp_store, idx)))
            if component == "mlp_out":
                hooks.append(layer.mlp.register_forward_hook(hook_maker(comp_store, idx)))
            if component == "layer_out":
                hooks.append(layer.register_forward_hook(hook_maker(comp_store, idx)))

    # --- shared forward / inference loop ---
    with torch.no_grad():
        num_batches = (len(str_inputs) + batch_size - 1) // batch_size

        for batch in tqdm.auto.tqdm(
            itertools.batched(str_inputs, n=batch_size),
            total=num_batches,
            desc=f"Inferring ({component or 'hidden'})",
        ):
            batch = list(batch)
            batch_inputs = tokenizer(batch, return_tensors="pt").to(model.device)

            # always request hidden states: needed for default behaviour
            out = model(**batch_inputs, output_hidden_states=True)

            # original behavior path (used when component=None)
            for li, h in enumerate(out.hidden_states):
                base_hidden[li].append(h[:, -1, :].detach().cpu())

            # predictions
            logits = out.last_hidden_state @ model.embed_tokens.weight.T
            next_ids = logits[:, -1, :].argmax(dim=-1)
            preds.extend(tokenizer.batch_decode(next_ids))

    # remove hooks if any
    for h in hooks:
        h.remove()

    # --- return values ----
    if component is None:
        hidden = {li: torch.cat(v, dim=0) for li, v in base_hidden.items()}
        return hidden, preds

    hidden = {}

    if component in ("q", "k", "v"):
        store = {"q": q_store, "k": k_store, "v": v_store}[component]
        for li, v in store.items():
            hidden[li] = torch.cat(v, dim=0)

    elif component == "qkv":
        for li in q_store.keys():
            hidden[li] = torch.cat([
                torch.cat(q_store[li], dim=0),
                torch.cat(k_store[li], dim=0),
                torch.cat(v_store[li], dim=0),
            ], dim=-1)

    else:  # "attn_out", "mlp_out", "layer_out"
        for li, v in comp_store.items():
            hidden[li] = torch.cat(v, dim=0)

    return hidden, preds

In [ ]:
COMPONENT = "q"
batch_size = 1024
# also tried with sin probes:
# train_hidden_states, train_preds = get_hidden_states_and_preds(
#         model,
#         [make_str_input(val, 0) for val in train_inputs] + [make_str_input(val, 1) for val in train_inputs] + [make_str_input(val, 2) for val in train_inputs],
#         batch_size
# )

# multiplication only:
# states_preds = (
#     [get_hidden_states_and_preds(model, [make_str_input(val, i) for val in train_inputs], batch_size) for i in range(num_templates)]
# )
# train_labels_ref = torch.tensor([x1 * x2 for _ in range(num_templates) for x1, x2 in train_inputs])

# multiplication+addition:
states_preds = (
    [get_hidden_states_and_preds(model, [make_str_input(val, i) for val in train_inputs], batch_size, component=COMPONENT) for i in range(num_templates)]
    +
    [get_hidden_states_and_preds(model, [make_str_input_add(val) for val in train_inputs_add], batch_size, component=COMPONENT)]
)
train_labels_ref = torch.tensor([x1 * x2 for _ in range(num_templates) for x1, x2 in train_inputs]
                                + [x1 + x2 for x1, x2 in train_inputs_add], device=device)

hidden_states_all = [x[0] for x in states_preds]
preds_all = [x[1] for x in states_preds]

# multiplication only:
# train_hidden_states = {k: torch.concat([hidden_states_all[i][k] for i in range(num_templates)]) for k in hidden_states_all[0].keys()}
# multiplication+addition:
train_hidden_states = {k: torch.concat([hidden_states_all[i][k] for i in range(num_templates+1)]) for k in hidden_states_all[0].keys()}

train_preds = list(itertools.chain(*preds_all))

In [ ]:
# val and test contain only multuplication in either case
valid_hidden_states, valid_preds = get_hidden_states_and_preds(
        model,
        [make_str_input(val) for val in valid_inputs],
        batch_size,
        component=COMPONENT,
)
valid_labels_ref = torch.tensor([x1 * x2 for x1, x2 in valid_inputs], device=device)
test_hidden_states, test_preds = get_hidden_states_and_preds(
        model,
        [make_str_input(val) for val in test_inputs],
        batch_size,
        component=COMPONENT,
)
test_labels_ref = torch.tensor([x1 * x2 for x1, x2 in test_inputs], device=device)

In [ ]:
train_inputs_t = torch.tensor(train_inputs)

train_inputs_t[:, 0] * train_inputs_t[:, 1]

In [ ]:
def sanitize_pred(pred: str) -> int:
    try:
        return int(pred)
    except ValueError:
        return -1

In [ ]:
test_inputs_t = torch.tensor(test_inputs)

train_preds_t = torch.tensor([sanitize_pred(pred) for pred in train_preds])
valid_preds_t = torch.tensor([sanitize_pred(pred) for pred in valid_preds])
test_preds_t = torch.tensor([sanitize_pred(pred) for pred in test_preds])

# ratio of properly extracted train predictions
sum(train_preds_t != -1) / len(train_preds_t)

In [ ]:
test_preds_t.device, test_labels_ref.device

In [ ]:
# absolute model accuracy on test set
sum(test_preds_t.to(device) == test_labels_ref) / len(test_inputs)

### Probing

In [ ]:
class ClassifierProbe(torch.nn.Module):
    basis: torch.Tensor

    def __init__(self, emb_dim: int, hidden_dim: int, heldout_mask: torch.Tensor):
        super().__init__()
        self.emb_to_latent = torch.nn.Linear(emb_dim, hidden_dim, bias=True)
        self.basis_to_latent = torch.nn.Linear(self.basis.shape[-1], hidden_dim, bias=True)
        self.basis = self.basis.to(device)
        self.heldout_mask: torch.nn.Buffer
        # self.register_buffer("basis", self.basis)
        self.register_buffer("heldout_mask", heldout_mask)
    def forward(self, x: Tensor, holdout_eval_tokens: bool) -> Tensor:
        latent_x = self.emb_to_latent(x)
        # during training, model learns to choose among only training tokens
        # but during eval, model must choose among all tokens
        # this means that the model is never exposed to the eval tokens during training
        latent_choices = self.basis_to_latent(self.basis)
        logits = latent_x @ latent_choices.T
        if holdout_eval_tokens:
            logits[:, self.heldout_mask] = float("-inf")
        return logits

In [ ]:
class SinProbe(ClassifierProbe):

    def __init__(self, *args, **kwargs):
        self.basis = sinusoidal_encode(torch.arange(1000), min_value=0, max_value=1000,
                                       embedding_dim=train_hidden_states[0].shape[-1])
        super().__init__(*args, **kwargs)

class BinProbe(ClassifierProbe):

    def __init__(self, *args, **kwargs):
        self.basis = binary_encode(torch.arange(1000), min_value=0, max_value=1000, embedding_dim=10)
        super().__init__(*args, **kwargs)


In [ ]:
class SinAccelClassifier(torch.nn.Module):
    def __init__(self, emb_dim: int, hidden_dim: int, choices: torch.Tensor, heldout_mask: torch.Tensor):
        super().__init__()
        self.emb_to_latent = torch.nn.Linear(emb_dim, hidden_dim, bias=True)
        self.freqs = torch.nn.Parameter(torch.linspace(1/(choices.max() - choices.min()), 0.5, steps=hidden_dim))
        self.phases = torch.nn.Parameter(torch.zeros(hidden_dim))
        self.amplitudes = torch.nn.Parameter(torch.zeros(hidden_dim))
        self.accels = torch.nn.Parameter(torch.zeros(hidden_dim))
        self.hidden_dim = hidden_dim
        self.heldout_mask: torch.nn.Buffer
        self.choices: torch.nn.Buffer
        self.register_buffer("heldout_mask", heldout_mask)
        self.register_buffer("choices", choices)

    def get_waves(self) -> Tensor:
        # USE THIS FORMULA
        waves = torch.sin(
            self.phases.unsqueeze(1)
            + (2 * torch.pi * self.freqs.unsqueeze(1) * self.choices.unsqueeze(0))
            + (2 * torch.pi * self.accels.unsqueeze(1) * torch.log(self.choices.unsqueeze(0) + 1e-4))
        )
        # sort by frequency
        waves = waves[torch.argsort(self.freqs.abs()), :]
        assert waves.shape == (self.hidden_dim, len(self.choices))
        return waves * self.amplitudes.unsqueeze(1)

    def forward(self, x: Tensor, holdout_eval_tokens: bool) -> Tensor:
        latent_x = self.emb_to_latent(x)
        waves = self.get_waves()
        logits = latent_x @ waves

        # during training, model learns to choose among only training tokens
        # but during eval, model must choose among all tokens
        # this means that the model is never exposed to the eval tokens during training
        if holdout_eval_tokens:
            logits[:, self.heldout_mask] = -torch.inf
        return logits

In [ ]:
# model's own predictions as labels

# train_labels = train_preds_t.detach().clone()
# train_hidden_states = {k: v[train_labels != -1] for k, v in train_hidden_states.items()}
# train_labels = train_labels[train_labels != -1]
#
# valid_labels = valid_preds_t.detach().clone()
# valid_hidden_states = {k: v[valid_labels != -1] for k, v in valid_hidden_states.items()}
# valid_labels = valid_labels[valid_labels != -1].to(device)
#
# test_labels = test_preds_t.detach().clone()
# test_hidden_states = {k: v[test_labels != -1] for k, v in test_hidden_states.items()}
# test_labels = test_labels[test_labels != -1].to(device)

In [ ]:
# true results as labels
train_labels = train_labels_ref
valid_labels = valid_labels_ref
test_labels = test_labels_ref

In [ ]:
# how many of the model outputs are valid numbers (=usable as labels in training)
sum(train_preds_t != -1) / len(train_preds_t)

In [ ]:
test_extracted = {}

test_accuracies = {"sin_accel": {}}

basis_name = "sin_accel"
batch_size = 1024

for component in {"q", "k", "v", "qkv", "attn_out", "mlp_out", "layer_out"}:
    # BLOCK: hidden states inference
    # multiplication+addition:
    states_preds = (
        [get_hidden_states_and_preds(model, [make_str_input(val, i) for val in train_inputs], batch_size, component=component) for i in range(num_templates)]
        +
        [get_hidden_states_and_preds(model, [make_str_input_add(val) for val in train_inputs_add], batch_size, component=component)]
    )
    train_labels_ref = torch.tensor([x1 * x2 for _ in range(num_templates) for x1, x2 in train_inputs]
                                    + [x1 + x2 for x1, x2 in train_inputs_add], device=device)

    hidden_states_all = [x[0] for x in states_preds]
    preds_all = [x[1] for x in states_preds]

    # multiplication only:
    # train_hidden_states = {k: torch.concat([hidden_states_all[i][k] for i in range(num_templates)]) for k in hidden_states_all[0].keys()}
    # multiplication+addition:
    train_hidden_states = {k: torch.concat([hidden_states_all[i][k] for i in range(num_templates+1)]) for k in hidden_states_all[0].keys()}

    train_preds = list(itertools.chain(*preds_all))

    # val and test contain only multuplication in either case
    valid_hidden_states, valid_preds = get_hidden_states_and_preds(
            model,
            [make_str_input(val) for val in valid_inputs],
            batch_size,
            component=component,
    )
    valid_labels_ref = torch.tensor([x1 * x2 for x1, x2 in valid_inputs], device=device)
    test_hidden_states, test_preds = get_hidden_states_and_preds(
            model,
            [make_str_input(val) for val in test_inputs],
            batch_size,
            component=component,
    )
    test_labels_ref = torch.tensor([x1 * x2 for x1, x2 in test_inputs], device=device)
    # END BLOCK: hidden states inference

    # for layer_idx in list(reversed(range(len(train_hidden_states)-2, len(train_hidden_states)))):
    for layer_idx in reversed(train_hidden_states.keys()):
        torch.manual_seed(0)

        probe = SinAccelClassifier(
            emb_dim=train_hidden_states[1].shape[-1],
            hidden_dim=500,
            choices=torch.arange(1000),
            heldout_mask=test_mask,
        ).to(device)

        optimizer = torch.optim.Adam(probe.parameters(), lr=1e-4, weight_decay=1e-3)
        # decaying LR is important for outputs probing and natural-lang contexts -- requiring inherently lower LR to fit
        scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.1, total_iters=15000)

        rng = torch.Generator().manual_seed(0)
        best_val_acc = -1
        best_ckpt = None
        for i in range(20000+1):
            probe.train()
            optimizer.zero_grad()
            minibatch_idcs = torch.randint(len(train_labels), size=(8,), generator=rng)
            x = train_hidden_states[layer_idx][minibatch_idcs].float().to(device)
            y = train_labels[minibatch_idcs].to(device)
            logits = probe(x, holdout_eval_tokens=True)
            # add l1 regularization of all params to the loss
            assert logits.shape[1] >= max(y)
            assert min(y) >= 0
            loss = torch.nn.functional.cross_entropy(logits, y)
            # loss += 0.001 * (probe.emb_to_latent.weight.abs().sum() + probe.amplitudes.abs().sum())
            loss.backward()
            optimizer.step()
            scheduler.step()
            if i % 100 == 0:
                train_acc = (logits.argmax(dim=-1) == y).float().mean().item()
                probe.eval()
                with torch.no_grad():
                    valid_logits = probe(valid_hidden_states[layer_idx].float().to(device), holdout_eval_tokens=False)
                    valid_loss = torch.nn.functional.cross_entropy(valid_logits, valid_labels)
                    valid_accuracy = (valid_logits.argmax(dim=-1) == valid_labels).float().mean().item()
                    if valid_accuracy > best_val_acc:
                        best_val_acc = valid_accuracy
                        best_ckpt = probe.state_dict()
                # print(f"{basis_name} {i=:>5} train loss: {loss.item():5.2f}  train acc: {train_acc:.2f}  val loss: {valid_loss.item():5.2f}  valid acc: {valid_accuracy:.2f}")
        probe.load_state_dict(best_ckpt)
        probe.eval()
        with torch.no_grad():
            test_logits = probe(test_hidden_states[layer_idx].float().to(device), holdout_eval_tokens=False)
            test_extracted[layer_idx] = test_logits.argmax(dim=-1)
            test_accuracy = (test_extracted[layer_idx] == test_labels).float().mean().item()

        test_accuracies[basis_name][layer_idx] = test_accuracy
        print(f"component {component} probe {basis_name} layer idx: {layer_idx:<3}, best valid accuracy: {best_val_acc:.2f}, test accuracy: {test_accuracy:.2f}")

In [ ]:
probe.accels

In [ ]:
# save the results for visualizations & further analyses
import pandas as pd

df = pd.DataFrame({"probe_l%s" % k: v.cpu() for k, v in test_extracted.items()})
df["model_predictions"] = test_preds_t.cpu()
df["inputs"] = [make_str_input(op) for op in test_inputs]
df["labels"] = test_labels_ref.cpu()

### Collecting the results of per-layer ablations

In [ ]:
for l_i in range(len(test_extracted)):
    model = transformers.AutoModel.from_pretrained(model_ckpt).eval().to(device)
    del model.layers[l_i]  # intervention -- dropping a single layer

    test_hidden_states, test_preds = get_hidden_states_and_preds(
        model,
        [make_str_input(val) for val in test_inputs],
        batch_size
    )
    test_preds_t = torch.tensor([sanitize_pred(pred) for pred in test_preds])

    # absolute model accuracy on test set
    df["model_predictions_without_l%s" % l_i] = test_preds_t.detach().cpu().tolist()

    acc = (sum(test_preds_t.to(device) == test_labels_ref) / len(test_inputs)).item()

    print("Acc without layer %s: %s" % (l_i, acc))

In [ ]:
# this output can be visualized using notebooks/viz/timothee_error_tracing.ipynb
df.to_csv("/home/xstefan3/tmp/pycharm_project_437/notebooks/logs/model_vs_probes_preds_llama3b_multi_271125.csv", index=False)

In [ ]:
# basic statistics of result recovery

is_result_computed_per_l = torch.vstack([test_extracted[l_key] == test_labels_ref for l_key in test_extracted])
is_result_computed_internally = torch.any(is_result_computed_per_l, dim=0).cpu()
returned_val_is_computed = torch.vstack([test_extracted[l_key].cpu() == test_preds_t for l_key in test_extracted])
is_result_returned = (test_preds_t == test_labels_ref.cpu())
print(
      "Ever computed correctly internally and NOT correctly returned (out of incorrect): %s\n"
      "NOT ever computed correctly internally and correctly returned: (out of correct) %s\n"
      "Ever computed correctly internally and correctly returned (out of correct): %s\n"
      "NOT ever computed correctly internally and NOT correctly returned (out of incorrect): %s\n"
      "Ever computed as returned: %s"
      % (
         torch.sum(is_result_computed_internally & ~is_result_returned).item() / (~is_result_returned).sum(),
         torch.sum(~is_result_computed_internally & is_result_returned).item() / is_result_returned.sum(),
         torch.sum(is_result_computed_internally & is_result_returned).item() / is_result_returned.sum(),
         torch.sum(~is_result_computed_internally & ~is_result_returned).item() / (~is_result_returned).sum(),
         returned_val_is_computed.any(dim=0).sum() / len(returned_val_is_computed[0]))
        )

In [ ]:
# ratios of cases with multiplication involving "1" or "2"

# in the case of Llama 1B, 15 out of 25 correct contains multiplication involving "1" or "2" --> effectively solvable by addition
# only 30 out of 140 in the case of Llama 3B
(len(test_labels_ref[~is_result_computed_internally & is_result_returned]),
torch.isin(test_inputs_t[~is_result_computed_internally & is_result_returned], torch.tensor([1, 2])).any(dim=1).sum())

In [ ]:
# baseline probes fitting (lin+log)

def solve_linear_layer(x: Tensor, y: Tensor) -> torch.nn.Linear:
    if y.ndim == 1:
        y = y.unsqueeze(-1)
    if not y.is_floating_point():
        y = y.float()
   
    lin = torch.nn.Linear(x.shape[-1], y.shape[-1], device=x.device)
    x_aug = torch.cat([x, torch.ones(len(x), 1, device=x.device)], dim=1)
    coeffs = torch.linalg.lstsq(x_aug, y).solution
    w, b = coeffs[:-1], coeffs[-1]
    with torch.no_grad():
        lin.weight[:] = w.T
        lin.bias[:] = b
    return lin

In [ ]:
test_accuracies["lin"] = {}
test_accuracies["log"] = {}

for layer_idx in range(1, len(train_hidden_states)):
    lin_probe = solve_linear_layer(
        train_hidden_states[layer_idx].float().to(device),
        train_labels.to(device),
    )
    log_probe = solve_linear_layer(
        train_hidden_states[layer_idx].float().to(device),
        train_labels.log1p().to(device),
    )
    lin_test_pred = lin_probe(test_hidden_states[layer_idx].float().to(device)).flatten().round().int()
    lin_test_accuracy = (lin_test_pred == test_labels).float().mean().item()
    
    log_test_pred = log_probe(test_hidden_states[layer_idx].float().to(device)).flatten().exp().add(1).round().int()
    log_test_accuracy = (log_test_pred == test_labels).float().mean().item()
    
    # test_accuracies["lin"][layer_idx] = lin_test_accuracy
    # test_accuracies["log"][layer_idx] = log_test_accuracy

    print(f"layer idx: {layer_idx:<3}, linear probe acc: {lin_test_accuracy:.2f}, log probe acc: {log_test_accuracy:.2f}")

In [ ]:
for name, accs in test_accuracies.items():
    print(f"{name} accs: | " + " | ".join([f"{x:.0%}" for layer, x in sorted(accs.items())]) + " |")